In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Performance Management: Qiskit Function od Q-CTRL Fire Opal
*Viz [referenci API](https://docs.quantum.ibm.com/api/functions/q-ctrl-performance-management)*

> **Note:** Qiskit Functions jsou experimentální funkce dostupné pouze uživatelům plánů IBM Quantum&reg; Premium Plan, Flex Plan a On-Prem (prostřednictvím IBM Quantum Platform API). Jsou ve stavu náhledové verze a mohou se změnit.


<Accordion>
<AccordionItem title="Package versions">

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit[all]~=2.3.1
qiskit-ibm-runtime~=0.45.1
```
</AccordionItem>
</Accordion>
## Přehled
Fire Opal Performance Management umožňuje komukoli dosáhnout smysluplných výsledků z kvantových počítačů ve velkém měřítku, aniž by bylo potřeba být expertem na kvantový hardware. Při spouštění Circuit pomocí Fire Opal Performance Management se automaticky uplatňují techniky potlačení chyb řízené umělou inteligencí, což umožňuje škálovat větší problémy s více Gate a qubit. Tento přístup snižuje počet snímků (shots) potřebných k dosažení správné odpovědi, bez jakékoli přidané režie – což přináší výrazné úspory jak ve výpočetním čase, tak v nákladech.

Performance Management potlačuje chyby a zvyšuje pravděpodobnost získání správné odpovědi na zašuměném hardwaru. Jinými slovy, zvyšuje poměr signálu k šumu. Následující obrázek ukazuje, jak zvýšená přesnost umožněná Performance Management může snížit potřebu dalších snímků v případě algoritmu kvantové Fourierovy transformace s 10 qubit. Pouze s 30 snímky Q-CTRL dosáhne prahové hodnoty spolehlivosti 99 %, zatímco výchozí nastavení (`QiskitRuntime` Sampler, `optimization_level`=3 a `resilience_level`=1, `ibm_sherbrooke`) vyžaduje 170 000 snímků. Tím, že získáš správnou odpověď rychleji, ušetříš výrazný výpočetní čas.

![Vizualizace vylepšeného výkonu](../docs/images/guides/qctrl-performance-management/achieve_more.svg)

Funkci Performance Management lze použít s jakýmkoli algoritmem a snadno ji lze použít místo standardních [Qiskit Runtime primitiv](/guides/primitives). V zákulisí spolupracuje více technik potlačení chyb, aby se předešlo chybám za běhu. Všechny metody pipeline Fire Opal jsou předem nakonfigurovány a jsou agnostické vůči algoritmu, což znamená, že vždy získáš nejlepší výkon ihned po spuštění.

Chceš-li získat přístup k Performance Management, [kontaktuj Q-CTRL](https://form.typeform.com/to/uOAVDnGg?typeform-source=q-ctrl.com).
## Popis
Fire Opal Performance Management nabízí dvě možnosti spuštění podobné primitivům Qiskit Runtime, takže snadno zaměníš standardní Sampler a Estimator za varianty Q-CTRL. Obecný postup použití funkce Performance Management je:
1. Definuj svůj Circuit (a operátory v případě Estimatoru).
2. Spusť Circuit.
3. Načti výsledky.

Pro snížení šumu hardwaru Fire Opal používá řadu technik potlačení chyb řízených umělou inteligencí zobrazených na následujícím obrázku. S Fire Opal je celý pipeline plně automatizovaný a nevyžaduje žádnou konfiguraci.

Pipeline Fire Opal eliminuje potřebu další režie, jako je zvýšený kvantový čas běhu nebo extra fyzické qubit. Všimni si, že klasický výpočetní čas zůstává faktorem (viz sekci [Benchmarky](#benchmarks) pro odhady, kde „Celkový čas" zahrnuje jak klasické, tak kvantové zpracování). Na rozdíl od mitigace chyb, která vyžaduje režii ve formě vzorkování, potlačení chyb Fire Opal pracuje jak na úrovni Gate, tak na úrovni pulzů, aby řešilo různé zdroje šumu a zabránilo vzniku chyb. Zabráněním chybám se eliminuje potřeba drahého post-processingu.

Následující obrázek zobrazuje metody potlačení chyb automatizované pomocí Fire Opal Performance Management.

![Vizualizace pipeline potlačení chyb](../docs/images/guides/qctrl-performance-management/error_suppression.svg)

Funkce nabízí dvě primitiva, Sampler a Estimator, a vstupy i výstupy obou rozšiřují implementovanou specifikaci pro [Qiskit Runtime V2 primitiva](/guides/primitive-input-output#pubs).
## Benchmarky
[Publikované výsledky algoritmického benchmarkování](https://journals.aps.org/prapplied/abstract/10.1103/PhysRevApplied.20.024034) prokazují výrazné zlepšení výkonu u různých algoritmů, včetně Bernstein-Vazirani, kvantové Fourierovy transformace, Groverova vyhledávání, kvantového aproximačního optimalizačního algoritmu a variačního kvantového eigensolverú. Zbytek této sekce poskytuje více podrobností o typech algoritmů, které lze spouštět, a o očekávaném výkonu a dobách běhu.

Následující nezávislé studie ukazují, jak Performance Management Q-CTRL umožňuje algoritmický výzkum v rekordním měřítku:
- [Parametrized Energy-Efficient Quantum Kernels for Network Service Fault Diagnosis](https://arxiv.org/abs/2405.09724v1) – kvantové kernelové učení až na 50 qubit
- [Tensor-based quantum phase difference estimation for large-scale demonstration](https://arxiv.org/abs/2408.04946) – kvantový odhad fáze až na 33 qubit
- [Hierarchical Learning for Quantum ML: Novel Training Technique for Large-Scale Variational Quantum Circuits](https://arxiv.org/abs/2311.12929) – kvantové načítání dat až na 21 qubit

Následující tabulka poskytuje přibližný přehled přesnosti a dob běhu z předchozích benchmarkových spuštění na `ibm_fez`. Výkon na jiných zařízeních se může lišit. Čas využití vychází z předpokladu 10 000 snímků na obvod. Uvedený „Počet qubit" není pevným omezením, ale představuje přibližné prahy, kde lze očekávat velmi konzistentní přesnost řešení. Větší velikosti problémů byly úspěšně vyřešeny a testování za těmito limity je vítáno.

| Příklad    | Počet qubit | Přesnost | Míra přesnosti | Celkový čas (s) | Využití Runtime (s) | Primitivum (Režim) |
| ---------  | ---------------- | -------------------------- | -------- | ---------- | ------------- |------------- |
| Bernstein–Vazirani  |  50Q    | 100%  | Míra úspěšnosti (procento běhů, kde správná odpověď má nejvyšší počet bitstring)     | 10    | 8         | Sampler |
| Kvantová Fourierova transformace | 30Q              | 100% | Míra úspěšnosti (procento běhů, kde správná odpověď má nejvyšší počet bitstring)      | 10    | 8        | Sampler |
| Kvantový odhad fáze  | 30Q   | 99,9998%  | Přesnost nalezeného úhlu: `1- abs(real_angle - angle_found)/pi`  | 10  | 8  | Sampler |
| Kvantová simulace: Isingův model (15 kroků) | 20Q  | 99,775%   |  $A$ (definováno níže)  |  60 (na krok)  | 15 (na krok)   | Estimator |
| Kvantová simulace 2: molekulární dynamika (20 časových bodů) | 34Q  |  96,78%  |  $A_{mean}$ (definováno níže)   | 10 (na časový bod)    | 6 (na časový bod)  | Estimator |

Definice přesnosti měření střední hodnoty – metrika $A$ je definována takto:
$$
A = 1 - \frac{|\epsilon^{ideal} - \epsilon^{meas}|}{\epsilon^{ideal}_{max} - \epsilon^{ideal}_{min}},
$$
kde $ \epsilon^{ideal} $ = ideální střední hodnota,  $ \epsilon^{meas} $ = naměřená střední hodnota, $\epsilon^{ideal}_{max} $ = ideální maximální hodnota a $\epsilon^{ideal}_{min}$ = ideální minimální hodnota. $A_{mean}$ je jednoduše průměr hodnoty $A$ přes více měření.

Tato metrika se používá, protože je invariantní vůči globálním posunům a škálování v rozsahu dosažitelných hodnot. Jinými slovy, bez ohledu na to, zda posunout rozsah možných středních hodnot nahoru nebo dolů nebo zvětšit rozptyl, hodnota $A$ by měla zůstat konzistentní.
## Začínáme
Fire Opal Performance Management používá Qiskit v`2.0.0`, což je doporučená verze. Podporované verze jsou Qiskit >=v`2.0.0`.
Ověř se pomocí svého [klíče IBM Quantum Platform API](http://quantum.cloud.ibm.com/) a vyber Qiskit Function takto. (Tento úryvek předpokládá, že jsi již [uložil/a svůj účet](/guides/functions#install-qiskit-functions-catalog-client) do svého lokálního prostředí.)

In [1]:
# This cell is hidden from users. It hides the "...You have imported samplomatic..." warning.
import warnings

warnings.filterwarnings("ignore", message=".*You have imported samplomatic*")

In [2]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# verify that you have access to the function
catalog.list()

[QiskitFunction(qunova/hivqe-chemistry),
 QiskitFunction(global-data-quantum/quantum-portfolio-optimizer),
 QiskitFunction(algorithmiq/tem),
 QiskitFunction(qedma/qesem),
 QiskitFunction(multiverse/singularity),
 QiskitFunction(ibm/circuit-function),
 QiskitFunction(q-ctrl/optimization-solver),
 QiskitFunction(colibritd/quick-pde),
 QiskitFunction(q-ctrl/performance-management),
 QiskitFunction(kipu-quantum/iskay-quantum-optimizer)]

In [3]:
# Access Function
perf_mgmt = catalog.load("q-ctrl/performance-management")

<Admonition type="note" title="Does this function support all IBM backends?">
If you want to use a backend that this function does not currently support, [reach out to Q-CTRL](https://form.typeform.com/to/iuujEAEI?typeform-source=q-ctrl.com) to add support.
</Admonition>

## Estimator primitive

### Estimator example

Use Fire Opal Performance Management's Estimator primitive to determine the expectation value of a single circuit-observable pair.

In addition to the `qiskit-ibm-catalog` and `qiskit` packages, you will also use the `numpy` package to run this example. You can install this package by uncommenting the following cell if you are running this example in a notebook using the IPython kernel.

In [4]:
# %pip install numpy

**2. Spusť Circuit**

Spusť Circuit a volitelně definuj Backend a počet snímků.

In [5]:
import numpy as np
from qiskit.circuit.library import iqp
from qiskit.quantum_info import random_hermitian, SparsePauliOp

n_qubits = 50

# Generate a random circuit
mat = np.real(random_hermitian(n_qubits, seed=1234))
circuit = iqp(mat)
circuit.measure_all()

# Define observables as a string
observable = SparsePauliOp("Z" * n_qubits)

In [6]:
# Create PUB tuple
estimator_pubs = [(circuit, observable)]

K ověření stavu své zátěže Qiskit Function můžeš použít dobře známá [Qiskit Serverless API](/guides/serverless):

In [7]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy().name

In [8]:
# Run the circuit using Estimator
qctrl_estimator_job = perf_mgmt.run(
    primitive="estimator",
    pubs=estimator_pubs,
    backend_name=backend_name,
)

**3. Načti výsledek**

In [9]:
qctrl_estimator_job.status()

'QUEUED'

Výsledky mají stejný formát jako [výstup Estimatoru](/guides/estimator-input-output):

In [10]:
# Retrieve the counts from the result list
result = qctrl_estimator_job.result()

The results have the same format as an [Estimator result](/docs/guides/estimator-input-output):

In [22]:
import numpy

result_str = str(result)

with numpy.printoptions(threshold=200):
    print(
        f"The result of the submitted job had {len(result)} PUB "
        f"and has a value:\n {result[0]}\n"
    )

print("The associated PubResult of this job has the following DataBins:")
print(f"{result[0].data}\n")

print(f"And this DataBin has attributes: {result[0].data.keys()}")

print("The expectation values measured from this PUB are:")
print(f"{result[0].data.evs}")

The result of the submitted job had 1 PUB
The result of the submitted job had 1 PUB and has a value:
 PubResult(data=DataBin(evs=0.0195, stds=0.9998098569228051), metadata={'precision': None})

The associated PubResult of this job has the following DataBins:
DataBin(evs=0.0195, stds=0.9998098569228051)

And this DataBin has attributes: dict_keys(['evs', 'stds'])
The expectation values measured from this PUB are:
0.0195


## Primitivum Sampler
### Příklad Sampleru
Použij primitivum Sampler Fire Opal Performance Management ke spuštění Circuit Bernstein–Vazirani. Tento algoritmus, používaný k nalezení skrytého řetězce z výstupů funkce černé skříňky, je běžným benchmarkovacím algoritmem, protože existuje jediná správná odpověď.
**1. Vytvoř Circuit**

Definuj správnou odpověď na algoritmus, skrytý bitstring, a Circuit Bernstein–Vazirani. Šířku Circuit lze snadno upravit pouhou změnou `circuit_width`.

In [12]:
import qiskit

circuit_width = 35
hidden_bitstring = "1" * circuit_width

# Create circuit, reserving one qubit for BV oracle
bv_circuit = qiskit.QuantumCircuit(circuit_width + 1, circuit_width)
bv_circuit.x(circuit_width)
bv_circuit.h(range(circuit_width + 1))
for input_qubit, bit in enumerate(reversed(hidden_bitstring)):
    if bit == "1":
        bv_circuit.cx(input_qubit, circuit_width)
bv_circuit.barrier()
bv_circuit.h(range(circuit_width + 1))
bv_circuit.barrier()
for input_qubit in range(circuit_width):
    bv_circuit.measure(input_qubit, input_qubit)

# Create PUB tuple
sampler_pubs = [(bv_circuit,)]

**2. Spusť Circuit**

Spusť Circuit a volitelně definuj Backend a počet snímků.

In [13]:
# Run the circuit using Sampler
qctrl_sampler_job = perf_mgmt.run(
    primitive="sampler",
    pubs=sampler_pubs,
    backend_name=backend_name,
)

Zkontroluj [stav](/guides/functions#check-job-status) své zátěže Qiskit Function nebo načti [výsledky](/guides/functions#retrieve-results) takto:

In [14]:
# Print the ID so you can use it later, if necessary
print(qctrl_sampler_job.job_id)

qctrl_sampler_job.status()

60fe2fa1-a860-43e4-8615-c6ac4180f93b


'QUEUED'

**3. Retrieve the result**

In [15]:
# Retrieve the job results
sampler_result = qctrl_sampler_job.result()

In [16]:
# Get results for the first (and only) PUB
pub_result = sampler_result[0]
counts = pub_result.data.c.get_counts()

print("Counts for the meas output register (limited to 30 results):")
for i, (bitstring, count) in enumerate(counts.items()):
    if i >= 50:
        print(f"  ... ({len(counts) - 30} more items)")
        break
    print(f"  {bitstring}: {count}")

Counts for the meas output register (limited to 30 results):
  11111111111111111111111111111111111: 1661
  11111111111111111111111111110111111: 60
  11111111111111111111111111111101111: 54
  11111111111111111111111111111110111: 54
  11111111111111011111111111111111111: 46
  11111111111111111110111111111111111: 44
  11111111111111111111111101111111111: 42
  11111111111111111111111110111111111: 42
  11111111111111110111111111111111111: 41
  11111111111111111111111111111111101: 39
  11111111111111111111101111111111111: 38
  11111111111111111111110111111111111: 38
  11111111111111111111111111101111111: 37
  11111111111111111111111111111111110: 36
  11111111111110111111111111111111111: 35
  11111111111111111111111111111011111: 32
  11111111111111101111111111111111111: 32
  01111111111111111111111111111111111: 27
  11111111111111111011111111111111111: 23
  11111111101111111111111111111111111: 22
  11111111111111111111111111111111011: 21
  11111111011111111111111111111111111: 20
  00000000000

**3. Plot the top bitstrings**

Plot the bitstring with the highest counts to see if the hidden bitstring was the mode.

In [17]:
import matplotlib.pyplot as plt


def plot_top_bitstrings(counts_dict, hidden_bitstring=None):
    # Sort and take the top 100 bitstrings
    top_100 = sorted(counts_dict.items(), key=lambda x: x[1], reverse=True)[
        :100
    ]
    if not top_100:
        print("No bitstrings found in the input dictionary.")
        return

    # Unzip the bitstrings and their counts
    bitstrings, counts = zip(*top_100)

    # Assign colors: purple if the bitstring matches hidden_bitstring,
    # otherwise gray
    colors = [
        "#680CE9" if bit == hidden_bitstring else "gray" for bit in bitstrings
    ]

    # Create the bar plot
    plt.figure(figsize=(15, 8))
    plt.bar(
        range(len(bitstrings)), counts, tick_label=bitstrings, color=colors
    )

    # Rotate the bitstrings for better readability
    plt.xticks(rotation=90, fontsize=8)
    plt.xlabel("Bitstrings")
    plt.ylabel("Counts")
    plt.title("Top 100 Bitstrings by Counts")

    # Show the plot
    plt.tight_layout()
    plt.show()

The hidden bitstring is highlighted in purple, and it should be the bitstring with the highest number of counts.

In [18]:
plot_top_bitstrings(counts, hidden_bitstring)

<Image src="../docs/images/guides/q-ctrl-performance-management/extracted-outputs/8106d906-0.avif" alt="Output of the previous code cell" />

**3. Vykresli nejčastější bitstringy**

Vykresli bitstringy s nejvyšším počtem výskytů a zjisti, zda byl skrytý bitstring nejčastějším.